# Customers UK View

Create a view of deduplicated customers located in the United Kingdom, using the golden source table.

**Lineage:** `CUSTOMERS_GOLDEN` → `CUSTOMERS_UK`

In [ ]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())[:8]
notebook_name = '02_customers_uk_view'
start_time = datetime.now()
print(f"Pipeline Run ID: {run_id}")

In [ ]:
%%sql -r create_uk_view
CREATE OR REPLACE VIEW LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_UK AS
SELECT
    GOLDEN_ID,
    CUSTOMER_ID,
    FULL_NAME,
    COMPANY_NAME,
    INVESTOR_TYPE,
    REGION,
    COUNTRY,
    AUM_COMMITMENT_GBP,
    RELATIONSHIP_START_DATE,
    RELATIONSHIP_MANAGER,
    RISK_PROFILE,
    EMAIL,
    STATUS
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN
WHERE IS_MASTER_RECORD = TRUE
  AND UPPER(COUNTRY) IN ('UK', 'UNITED KINGDOM', 'GREAT BRITAIN', 'ENGLAND', 'SCOTLAND', 'WALES', 'NORTHERN IRELAND')

In [ ]:
%%sql -r verify_uk
SELECT COUNT(*) AS uk_customer_count
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_UK

## Record Lineage

In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime

session = get_active_session()
end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

row_count = session.sql("SELECT COUNT(*) AS cnt FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_UK").collect()[0]['CNT']

session.sql(f"""
    INSERT INTO LANSDOWNEPARTNERS_DB.CORE.PIPELINE_LINEAGE 
    (RUN_ID, NOTEBOOK_NAME, STEP_NAME, SOURCE_OBJECT, TARGET_OBJECT, OPERATION, ROW_COUNT, STATUS, DURATION_SECONDS)
    VALUES (
        '{run_id}',
        '{notebook_name}',
        'create_uk_view',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_UK',
        'CREATE VIEW',
        {row_count},
        'SUCCESS',
        {duration}
    )
""").collect()

print(f"Lineage recorded: CUSTOMERS_GOLDEN -> CUSTOMERS_UK ({row_count} rows, {duration}s)")